# Sentinel-2 Time Series Download — Part 2 (remaining ~1,200 sites)

Complements `kaggle_sentinel2_download.ipynb` by downloading **all the sites Part 1 didn't**.

## What Part 1 covered
- All 678 KL-labeled test images
- 472 stratified-by-year train images (random_state=42)
- Total: 1,150 sites → ~18 GB zip

## What this notebook covers
- The remaining 1,200 non-KL train images
- Uses the same `random_state=42` so the 'already-picked' set matches Part 1 exactly

## How to use
Same as Part 1:
1. Upload this `.ipynb` to a **new** Kaggle notebook (don't re-use the Part 1 one)
2. Settings → Internet: **On**
3. **Save Version → Save & Run All (Commit)**
4. Close tab, come back in ~2 hours
5. Output tab → download `s2_stacks_part2.zip`
6. Unzip locally into `data/features/sentinel2/kaggle_run/` (merges with Part 1)

In [ ]:
!pip install -q pystac-client planetary-computer

In [ ]:
import os, sys, json, math, logging, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.enums import Resampling
from rasterio.warp import transform as warp_transform
from rasterio.windows import from_bounds

import pystac_client
import planetary_computer

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    force=True)

OUT_ROOT = '/kaggle/working/data/features/sentinel2'
os.makedirs(OUT_ROOT, exist_ok=True)

S2_BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
MPC_KEY  = {'B2':'B02','B3':'B03','B4':'B04','B5':'B05','B6':'B06',
            'B7':'B07','B8':'B08','B8A':'B8A','B11':'B11','B12':'B12'}
GOOD_SCL = {4, 5, 6, 7, 11}
AOI_HALF_M = 500
TARGET_SIZE = 100

print('Output root:', OUT_ROOT)

## Fetch label CSV and compute the COMPLEMENT of Part 1's selection

In [ ]:
CSV_URL = ('https://raw.githubusercontent.com/AIscend-Research/'
           'smallholder-irrigation-dataset/main/'
           'data/labels/labeled_surveys/random_sample/latest_irrigation_table.csv')
CSV_LOCAL = '/kaggle/working/latest_irrigation_table.csv'
r = requests.get(CSV_URL, timeout=60); r.raise_for_status()
with open(CSV_LOCAL, 'wb') as f: f.write(r.content)
df = pd.read_csv(CSV_LOCAL)

# Dedupe + flag KL-test images (same logic as Part 1)
uniq = df.drop_duplicates(subset=['site_id','year','month','day']).copy()
kl_keys = set(df[df['operator_initials']=='KL']
              .apply(lambda r: (r['site_id'], r['year'], r['month'], r['day']), axis=1))
uniq['has_kl'] = uniq.apply(lambda r: (r['site_id'], r['year'], r['month'], r['day']) in kl_keys, axis=1)

# Reproduce Part 1's stratified pick (same BUDGET, same random_state=42) so we know
# exactly which non-KL rows it grabbed — then take everything else.
BUDGET_PART1 = 1150
kl_set = uniq[uniq['has_kl']]
rest   = uniq[~uniq['has_kl']]
remaining_budget = max(0, BUDGET_PART1 - len(kl_set))

part1_train = rest.groupby('year', group_keys=False).apply(
    lambda g: g.sample(
        n=min(len(g), max(1, int(round(remaining_budget * len(g) / len(rest))))),
        random_state=42)
)
if len(part1_train) > remaining_budget:
    part1_train = part1_train.sample(n=remaining_budget, random_state=42)

part1_keys = set(part1_train.apply(
    lambda r: (r['site_id'], r['year'], r['month'], r['day']), axis=1))

# Part 2 = everything in `rest` that Part 1 did NOT pick
selected = rest[~rest.apply(
    lambda r: (r['site_id'], r['year'], r['month'], r['day']) in part1_keys, axis=1)
].reset_index(drop=True)

print(f'Part 1 KL test:       {len(kl_set)}')
print(f'Part 1 train pick:    {len(part1_train)}')
print(f'Part 1 total:         {len(kl_set)+len(part1_train)}')
print(f'Part 2 (this run):    {len(selected)}')
print(f'Grand total covered:  {len(kl_set)+len(part1_train)+len(selected)} / {len(uniq)} unique images')
print('Part 2 year distribution:')
print(selected['year'].astype(int).value_counts().sort_index())

## STAC download helpers (same as Part 1)

In [ ]:
_catalog = None
def get_catalog():
    global _catalog
    if _catalog is None:
        _catalog = pystac_client.Client.open(
            'https://planetarycomputer.microsoft.com/api/stac/v1',
            modifier=planetary_computer.sign_inplace,
        )
    return _catalog

def aoi_bounds_utm(lat, lon, dst_crs):
    xs, ys = warp_transform('EPSG:4326', dst_crs, [lon], [lat])
    return (xs[0]-AOI_HALF_M, ys[0]-AOI_HALF_M, xs[0]+AOI_HALF_M, ys[0]+AOI_HALF_M)

def search_best_item(lat, lon, start, end):
    cat = get_catalog()
    items = list(cat.search(
        collections=['sentinel-2-l2a'],
        intersects={'type':'Point','coordinates':[lon, lat]},
        datetime=f'{start}/{end}',
    ).items())
    if not items: return None
    items.sort(key=lambda it: it.properties.get('eo:cloud_cover') or math.inf)
    return items[0]

def read_band(href, bounds_utm, resampling):
    with rasterio.open(href) as src:
        win = from_bounds(*bounds_utm, transform=src.transform)
        arr = src.read(1, window=win, out_shape=(TARGET_SIZE, TARGET_SIZE),
                       resampling=resampling, boundless=True, fill_value=0)
        win_t = src.window_transform(win)
        sx, sy = win.width/TARGET_SIZE, win.height/TARGET_SIZE
        return arr.astype(np.uint16), win_t * rasterio.Affine.scale(sx, sy), src.crs

def write_stack(path, stack, transform, crs):
    bands, h, w = stack.shape
    with rasterio.open(path, 'w', driver='GTiff', height=h, width=w, count=bands,
                       dtype='uint16', crs=crs, transform=transform, nodata=0,
                       compress='deflate') as dst:
        dst.write(stack)

def export_one_scene(lat, lon, start, end, file_name, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    item = search_best_item(lat, lon, start, end)
    if item is None: return False
    with rasterio.open(item.assets['B02'].href) as s: scene_crs = s.crs
    bounds = aoi_bounds_utm(lat, lon, scene_crs)
    try:
        bands, tfm, crs = [], None, None
        for b in S2_BANDS:
            arr, t, c = read_band(item.assets[MPC_KEY[b]].href, bounds, Resampling.bilinear)
            bands.append(arr)
            if tfm is None: tfm, crs = t, c
        scl, _, _ = read_band(item.assets['SCL'].href, bounds, Resampling.nearest)
    except Exception as e:
        logging.error(f'read failed {item.id}: {e}'); return False
    unmasked = np.stack(bands, axis=0)
    good = np.isin(scl, list(GOOD_SCL))
    masked = unmasked * good[np.newaxis,:,:].astype(np.uint16)
    write_stack(os.path.join(out_dir, f'{file_name}.tif'), unmasked, tfm, crs)
    write_stack(os.path.join(out_dir, f'{file_name}_masked.tif'), masked, tfm, crs)
    return True

def retrieve_time_series(file_id, lat, lon, date, out_dir,
                          start_month=1, num_windows=36, timestep=10, window_buffer=3):
    step = timedelta(days=timestep)
    n_buf = num_windows + window_buffer*2
    start_d = datetime(date.year, start_month, 1) - step*window_buffer
    wins = [(start_d + i*step, start_d + (i+1)*step) for i in range(n_buf)]
    tmp = os.path.join(out_dir, '_tmp', file_id); os.makedirs(tmp, exist_ok=True)

    def paths(s, e):
        nm = f's2_{lat:.2f}_{lon:.2f}_{s.strftime("%Y-%m-%d")}_{e.strftime("%Y-%m-%d")}'
        return os.path.join(tmp, nm+'.tif'), os.path.join(tmp, nm+'_masked.tif'), s.strftime('%Y-%m-%d'), e.strftime('%Y-%m-%d')

    with ThreadPoolExecutor(max_workers=10) as ex:
        futs = []
        for s, e in wins:
            u, m, ss, es = paths(s, e)
            if not (os.path.exists(u) and os.path.exists(m)):
                nm = f's2_{lat:.2f}_{lon:.2f}_{ss}_{es}'
                futs.append(ex.submit(export_one_scene, lat, lon, ss, es, nm, tmp))
        for f in as_completed(futs):
            try: f.result()
            except Exception as e: logging.error(f'export err: {e}')

    empty = np.zeros((len(S2_BANDS), TARGET_SIZE, TARGET_SIZE), dtype=np.uint16)
    im_u, im_m, meta_w = [], [], []
    tcrs = ttf = None
    for s, e in wins:
        u, m, ss, es = paths(s, e)
        if os.path.exists(u) and os.path.exists(m):
            try:
                with rasterio.open(u) as src:
                    arr_u = src.read()
                    if tcrs is None: tcrs, ttf = src.crs, src.transform
                with rasterio.open(m) as src:
                    arr_m = src.read()
                im_u.append(arr_u); im_m.append(arr_m)
                frac = float(((arr_m==0).any(axis=0)).sum()/(TARGET_SIZE*TARGET_SIZE))
                meta_w.append({'date_range':[ss,es],'file_exists':True,'masked_fraction':frac})
            except Exception as e:
                logging.error(f'read fail: {e}')
                im_u.append(empty.copy()); im_m.append(empty.copy())
                meta_w.append({'date_range':[ss,es],'file_exists':False,'masked_fraction':1.0})
        else:
            im_u.append(empty.copy()); im_m.append(empty.copy())
            meta_w.append({'date_range':[ss,es],'file_exists':False,'masked_fraction':1.0})

    if tcrs is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError(f'No valid images for {file_id}')

    su = np.stack(im_u, axis=0); sm = np.stack(im_m, axis=0)
    T, B, H, W = su.shape
    for path, arr in [(f'{out_dir}/{file_id}_stack.tif', su.transpose(1,0,2,3).reshape(T*B, H, W)),
                      (f'{out_dir}/{file_id}_stack_masked.tif', sm.transpose(1,0,2,3).reshape(T*B, H, W))]:
        with rasterio.open(path, 'w', driver='GTiff', height=H, width=W, count=T*B,
                           dtype=su.dtype, crs=tcrs, transform=ttf, nodata=0,
                           compress='deflate') as dst:
            dst.write(arr)

    meta = {'file_id': file_id,'lat': float(lat),'lon': float(lon),
            'year': int(date.year),'collection':'L2A','bands': S2_BANDS,
            'shape': list(sm.shape),'target_size': TARGET_SIZE,
            'num_windows': num_windows,'num_windows_buffered': n_buf,
            'timestep_days': 10,'start_month_unbuffered': start_month,
            'window_buffer': window_buffer,'windows': meta_w}
    with open(f'{out_dir}/{file_id}_metadata.json','w') as f: json.dump(meta, f, indent=2)
    shutil.rmtree(tmp, ignore_errors=True)

## Run download (resume-safe)

In [ ]:
VERSION = 'kaggle_run'      # Same folder name as Part 1 so they merge on disk
out_dir = os.path.join(OUT_ROOT, VERSION)
os.makedirs(out_dir, exist_ok=True)

MAX_CONCURRENT_SITES = 3

def process_row(row):
    lat, lon = row['y'], row['x']
    date = datetime(int(row['year']), int(row['month']), int(row['day']))
    sid = str(row['site_id']).replace('id_','')
    file_id = f"{sid}_{date.year}.{date.month:02d}.{date.day:02d}"
    if os.path.exists(f'{out_dir}/{file_id}_stack.tif'):
        return 'skip', file_id, None
    try:
        retrieve_time_series(file_id, lat, lon, date, out_dir)
        return 'done', file_id, None
    except Exception as e:
        return 'fail', file_id, str(e)

rows = [r for _, r in selected.iterrows()]
total = len(rows)
done = skipped = failed = 0
logging.info(f'Starting Part 2: {total} sites, {MAX_CONCURRENT_SITES} concurrent')

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_SITES) as ex:
    futs = {ex.submit(process_row, r): r for r in rows}
    for f in as_completed(futs):
        status, fid, err = f.result()
        n = done + skipped + failed + 1
        if   status == 'done': done += 1;    logging.info(f'[{n}/{total}] done {fid}')
        elif status == 'skip': skipped += 1; logging.info(f'[{n}/{total}] skip {fid}')
        else:                  failed += 1;  logging.error(f'[{n}/{total}] FAIL {fid}: {err}')

logging.info(f'Finished Part 2: {done} done, {skipped} skipped, {failed} failed')

## Zip for download

In [ ]:
import subprocess
ZIP_PATH = '/kaggle/working/s2_stacks_part2.zip'
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
subprocess.run(['zip', '-r', '-q', ZIP_PATH, out_dir])
size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f'Zip ready: {ZIP_PATH}  ({size_mb:.0f} MB)')

## After both notebooks finish

1. Download `s2_stacks.zip` from Part 1 notebook → unzip into `data/features/sentinel2/kaggle_run/`
2. Download `s2_stacks_part2.zip` from this notebook → unzip into the **same** `data/features/sentinel2/kaggle_run/` (they merge cleanly because filenames are unique)
3. Total: all 2,350 unique sites locally
4. Proceed to Step 2 of Phase 1 (`create_label_band.py`)